# 📍 Chapter 01: Spatial DBMS Geodesy, Projections & Datum Shifts
## Managing Ghana's Tripartite Datum Conflict in PostGIS

### 🎯 The Mission: Why We Are Doing This
In Ghana, urban datasets arrive in three conflicting geodetic standards:
1. **WGS 84 (`EPSG:4326`)**: Raw GPS coordinates in angular degrees ($^\circ$).
2. **UTM Zone 30N (`EPSG:32630`)**: Ghana's national engineering standard in linear metric meters ($	ext{m}$).
3. **Ghana National Grid / War Office (`EPSG:2136`)**: Historical cadastral survey grid used by the Lands Commission.

If you measure distances or areas in `EPSG:4326`, PostGIS calculates in degrees (where $1^\circ pprox 111	ext{ km}$ at the equator), producing completely invalid numbers. As a Spatial DBA, you must enforce metric coordinate reprojection using `ST_Transform`.

---

### 🔑 Spatial DBA Keywords Decoded

| Function / Term | Plain-English Explanation | Production Trap / Note |
| :--- | :--- | :--- |
| **`ST_SetSRID(geom, srid)`** | Stamps an SRID tag onto geometry **WITHOUT** changing its $(X, Y)$ coordinate values. | Use ONLY when importing raw untagged coordinates. |
| **`ST_Transform(geom, target_srid)`** | **Reprojects** coordinates from one datum to another by recalculating the math. | Use to convert WGS84 GPS degrees to UTM 30N meters. |
| **`GIST` (Generalized Search Tree)** | Balanced $R$-Tree spatial index that speeds up spatial searches from $O(N)$ to $O(\log N)$. | Never run spatial queries without a GIST index! |
| **Bounding Box (`&&`)** | 2D rectangular overlap operator. Fast primary filter. | Evaluated in milliseconds using the spatial index. |

---

### ⚠️ The Professor's Pitfall & DBA Warning
> **The `ST_SetSRID` Disaster:** Never call `ST_SetSRID(geom_wgs84, 32630)`. This tells PostGIS that $(-0.18, 5.65)$ are UTM meters! Your geometry will be displaced into the Atlantic Ocean near $(0, 0)$. Always call `ST_Transform(geom_wgs84, 32630)`.

In [ ]:
from config_db import execute_sql, query_df, query_gdf, preview_map

# 1. Create table and insert Accra landmarks
sql_create = """
DROP TABLE IF EXISTS accra_landmarks CASCADE;
CREATE TABLE accra_landmarks (
    landmark_id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    category VARCHAR(50),
    geom_wgs84 GEOMETRY(Point, 4326),
    geom_utm30n GEOMETRY(Point, 32630),
    geom_ghana_grid GEOMETRY(Point, 2136)
);

INSERT INTO accra_landmarks (name, category, geom_wgs84)
VALUES 
    ('Kotoka International Airport (Terminal 3)', 'Transport', ST_SetSRID(ST_MakePoint(-0.1667, 5.6052), 4326)),
    ('University of Ghana (Balme Library, Legon)', 'Education', ST_SetSRID(ST_MakePoint(-0.1870, 5.6508), 4326)),
    ('Jamestown Lighthouse', 'Heritage', ST_SetSRID(ST_MakePoint(-0.2132, 5.5345), 4326)),
    ('Tema Port (Meridian Point)', 'Maritime', ST_SetSRID(ST_MakePoint(0.0000, 5.6350), 4326)),
    ('Accra Mall (Tetteh Quarshie)', 'Commercial', ST_SetSRID(ST_MakePoint(-0.1772, 5.6178), 4326)),
    ('Korle Bu Teaching Hospital', 'Healthcare', ST_SetSRID(ST_MakePoint(-0.2285, 5.5372), 4326));

-- Transform to UTM Zone 30N and Ghana Grid
UPDATE accra_landmarks
SET geom_utm30n     = ST_Transform(geom_wgs84, 32630),
    geom_ghana_grid = ST_Transform(geom_wgs84, 2136);

CREATE INDEX idx_landmarks_utm30n ON accra_landmarks USING GIST(geom_utm30n);
"""

execute_sql(sql_create)
print("✅ Landmarks inserted, transformed, and indexed.")

In [ ]:
# 2. Compare Planar vs Ellipsoidal Distances
df_distances = query_df("""
SELECT 
    a.name AS origin,
    b.name AS destination,
    ROUND(ST_Distance(a.geom_utm30n, b.geom_utm30n)::NUMERIC, 2) AS planar_dist_m,
    ROUND(ST_Distance(a.geom_wgs84::geography, b.geom_wgs84::geography)::NUMERIC, 2) AS ellipsoidal_dist_m,
    ROUND(ABS(ST_Distance(a.geom_utm30n, b.geom_utm30n) - ST_Distance(a.geom_wgs84::geography, b.geom_wgs84::geography))::NUMERIC, 3) AS distortion_m
FROM accra_landmarks a
CROSS JOIN accra_landmarks b
WHERE a.landmark_id < b.landmark_id
ORDER BY planar_dist_m DESC;
""")

df_distances

In [ ]:
# 3. Interactive Map Preview
gdf_landmarks = query_gdf("""
SELECT landmark_id, name, category, geom_utm30n AS geom FROM accra_landmarks;
""")

preview_map(gdf_landmarks, tooltip_cols=["name", "category"], zoom_start=11)